# Ceramic Network

WHAT IS THE QUESTION?? 

Does the two network share knowledge? or when this knowledge appeared? when? how to tell that? 

## I. Introduction and prior knowledge

Kroon, E.J. in his PhD thesis tried to reconstruct the interaction between two cultures: Funnel Beaker West (4000-2800 BCE)  and Corded Ware (3000-2350 BCE). In his thesis he uses a probabilistic approach and to determine the similarity he uses the wasserstein distance between the empirical pdfs of the edges, because from the dataset we can determine the probability to do a step after anoter in a certain culture: 
$p(x_{t+1}| x_{t})$.  

In [ ]:
from functions import *

#### I.1 Datasets

The datasets are as follows: 
* Rows --> Pot ID
* Cols, separated by ";" --> Pot work-chain steps (es. Wet clay ; Modelling ; Pressure ; Wet smoothing ; Leather-hard ; Dry ; Open firing)


**Comments on datasets:** 
* We have words instead of numbers, find embedding to have something quantitative? 
* The pots do not have any "metavariables" like the geographical position where they were found. That actually could be usefull, because we would have a constrain and a distance. 
* We do not have a theoretical model that explains the variabilities in the paths. Maximum caliber, which is like maximum entropy for the paths? 
* How to look at the path similarities? 
* Different path lengths

# Meeting with Dionigi (08/05/2025)

During the meeting has been proposed the following idea: recreate a network where the nodes are a set of 3 techniques. This should , in principle, identify a unique set of teqniques used by a culture. 

In [ ]:
graph_options = {
    'show_details': True,
    'show_details_toggle_button': True,
    'show_menu': True,
    'show_menu_toggle_button': True,
    'show_node': True,
    'node_size_factor': 1.2,
    'node_size_data_source': 'size',
    'use_node_size_normalization': False,
    'node_size_normalization_min': 10.0,
    'node_size_normalization_max': 50.0,
    'node_drag_fix': True,
    'node_hover_neighborhood': True,
    'node_hover_tooltip': True,
    'show_node_image': True,
    'node_image_size_factor': 1.0,
    'show_node_label': True,
    'show_node_label_border': False,
    'node_label_data_source': 'id',
    'node_label_size_factor': 0.8,
    'node_label_rotation': 45.0,
    'node_label_font': 'Arial',
    'show_edge': True,
    'edge_size_factor': 1.0,
    'edge_size_data_source': 'size',
    'use_edge_size_normalization': False,
    'edge_size_normalization_min': 0.2,
    'edge_size_normalization_max': 5.0,
    'edge_curvature': 0.0,
    'edge_hover_tooltip': True,
    'show_edge_label': True,
    'show_edge_label_border': False,
    'edge_label_data_source': 'label',
    'edge_label_size_factor': 1.0,
    'edge_label_rotation': 45.0,
    'edge_label_font': 'Arial',
    'zoom_factor': 0.4,
    'large_graph_threshold': 500,

    # Specific to d3 layout
    'use_many_body_force': True,
    'many_body_force_strength': -70.0,
    'many_body_force_theta': 0.9,
    'use_many_body_force_min_distance': False,
    'many_body_force_min_distance': 10.0,
    'use_many_body_force_max_distance': False,
    'many_body_force_max_distance': 1000.0,
    'use_links_force': True,
    'links_force_distance': 50.0,
    'links_force_strength': 0.5,
    'use_collision_force': True,
    'collision_force_radius': 30.0,
    'collision_force_strength': 0.9,
    'use_x_positioning_force': False,
    'x_positioning_force_strength': 0.2,
    'use_y_positioning_force': True,
    'y_positioning_force_strength': 0.5,
    'use_centering_force': True,
}


In [ ]:
from functions import *
data_FBW = pd.read_csv('./data/FBW.csv', header=None, sep=';')
G = graph_from_data(data_FBW)
gv.d3(G)

In [ ]:
G_clustered = cluster_nodes(data_FBW, num_data = 0, coloring= 'cool')
#gv.d3(G_clustered, show_edge_label=True, edge_label_data_source = 'label')
gv.d3(G_clustered, **graph_options)

### Considerations: 
* Grouping the nodes in this way **increases the complexity of the network** adding more nodes than the one containing only the single step. (A better clustering/renormalisation?)
* Difficulty to find an appropriate model that can describe the path creation and identification. 
* **Not working with real or natural numbers, but with words...** some type of embedding for similarity? 
* Can we consider a **work-chain as a time series?** Try maximum entropy approach? Possible problems: If they were timeseries they are **short** and **not of the same length**.  
* What information can we extract? Can we construct **correlation matrices**? 
* Possible question: Does dataset A explain part of dataset B? Do we need a null-model? (in the thesis he found a null model but only for the single nodes)


# II. Data Visualisation

In this section we will explore the datasets and their modifications to get different point of views on the data

## II.1 Network reduction 

Once we plotted all the network some conseutive nodes will still have degree=2, this means that those subset of nodes is not important in relations between other nodes and can be collapsed as a unique node (colored differently). 

In [ ]:
G_contracted, components, contracted_nodes = contract_low_degree_nodes(G_clustered, position = 'flow', remove_chain =False)
gv.d3(G_contracted, **graph_options, layout_algorithm_active=False)

In [ ]:
G_contracted, components, contracted_nodes = contract_low_degree_nodes(G_clustered, position = 'flow', remove_chain = True)
gv.d3(G_contracted, **graph_options, layout_algorithm_active=False)

## II.2 Empirical PDFs 

Here we plot the empirical pdfs obtained with the empirical edge probabilities. We first start with the dataset without triplet contractions. Note that both the datasets share the same work-steps, but the transmission of a knowledge is suppose to happen when you share a sub-sequence rather than a single technique. 

The pdf, is the empirica probability distribution of the paths. This is calculated based on the edges and on their weights that define the transition probability from one techniques to another.

In [ ]:
data_CW = pd.read_csv('./data/CW.csv', header=None, sep=';')
data_CW.head()

### II.2.1 Raw data pdf similarity

In [ ]:
G_CW = graph_from_data(data_CW)
G_FBW = graph_from_data(data_FBW)
start_node = 'Wet clay'
end_node = 'Open firing'
most_prob_path = find_most_probable_path(G_FBW, start_node, end_node)
print(f"Most probable path from '{start_node}' to '{end_node}':")
for node in most_prob_path:
    print(node)
# color the path and the edges in the path
for node in most_prob_path:
    G_FBW.nodes[node]['color'] = 'orange'
# color the connecting edges
for i in range(len(most_prob_path) - 1):
    edge = (most_prob_path[i], most_prob_path[i + 1])
    G_FBW.edges[edge]['color'] = 'orange'

G_FWD = add_positions(G_FBW, position='flow')
gv.d3(G_FBW, **graph_options, layout_algorithm_active=False)

Find the empirical path distribution: 
$$ P( \Gamma =\{x_i\}_i) = P(x_n|x_{n-1}) ... P(x_1|x_0)P(x_0) $$

where the $P(x_n|x_{n-1})$ is the edge-weight and $x_i$ is the node/step/triplet. 

However, I have one doubt wheter one can do this because the **paths have different lengths**

In [ ]:
path_probabilities = path_pdf_weights(data_FBW, )
# random check: normalization
s = sum(path_probabilities.values())
print(f"Sum of path probabilities: {s}")
path_prob_fig = plot_path_pdf(path_probabilities, show=True)

In [ ]:
path_probabilities = path_pdf_weights(data_CW)
path_prob_fig = plot_path_pdf(path_probabilities, show=False)
path_prob_fig.show()

In [ ]:
path_probabilities_FBW = path_pdf_weights(data_FBW)
path_probabilities_CW = path_pdf_weights(data_CW)
path_comparison_fig = plot_path_abundance_comparison(path_probabilities_FBW, path_probabilities_CW, name1="FBW Dataset", name2="CW Dataset", show=True)

### II.2.2 Triplet contraction Prob_distr_funct similarity

We do not know the new probabilities!?!?!

In [ ]:
G_clustered_FBW = cluster_nodes(data_FBW, num_data=0, coloring='cool', type_edge_weight = 'probability')

path_probabilities_FBW_clustered, G_FBW_clustered = path_pdf_clustered_graph(data_FBW, save_paths_with_clustered_nodes = True, filename = "./data/clustered_paths_FBW")

path_prob_fig_FBW_clust = plot_path_pdf(path_probabilities_FBW_clustered, show=True)


In [ ]:
G_clustered_CW = cluster_nodes(data_CW, num_data=0, coloring='cool', type_edge_weight = 'probability')

path_probabilities_CW_clustered, G_CW_clustered = path_pdf_clustered_graph(data_CW, save_paths_with_clustered_nodes = True, filename = "./data/clustered_paths_CW")

path_prob_fig_CW_clust = plot_path_pdf(path_probabilities_CW_clustered, show=True)

In [ ]:
path_comparison_fig = plot_path_abundance_comparison(path_probabilities_FBW_clustered, path_probabilities_CW_clustered, name1="FBW Dataset", name2="CW Dataset", show=True)
# ploty save image 
path_comparison_fig.write_image("RW_archeo/path_abundance_comparison.pdf")

In [ ]:
def run_manual_random_walk(G_CW_clustered, initial_nodes, ending_nodes, *, iterations=1000, mode="no"):
    import random

    def _edge_key(u, v):
        return (u, v)

    def init_edge_weights(G: nx.DiGraph, weight_attr="edge_counting", default=1.0):
        base_w = {}
        for u, v, d in G.edges(data=True):
            base_w[(u, v)] = float(d.get(weight_attr, default))
            print(base_w)
        return base_w

    transition_prob = nx.get_edge_attributes(G_CW_clustered, "edge_counting")
    for (u, v), weight in transition_prob.items():
        print(f"Edge: {u} -> {v}, Initial weight: {weight}")
    walk = []
    for _ in range(iterations):
        tmp = []
        zero_indegree_nodes = [node for node, degree in G_CW_clustered.in_degree() if degree == 0]
        idx = np.random.choice(
            len(zero_indegree_nodes),
            p=list(initial_nodes.get(node, 0) for node in zero_indegree_nodes)
            / np.sum(list(initial_nodes.get(node, 0) for node in zero_indegree_nodes)),
        )
        tmp.append(zero_indegree_nodes[idx])
        while G_CW_clustered.out_degree(tmp[-1]) != 0:
            current_node = tmp[-1]
            if current_node not in G_CW_clustered:
                break
            if current_node in ending_nodes.keys() and random.random() > ending_nodes[current_node]:
                break
            neighbors = list(G_CW_clustered.successors(current_node))
            print(f"Current node: {current_node}, Neighbors: {neighbors}")
            norm = np.sum(list(transition_prob[(current_node, neighbor)] for neighbor in neighbors))
            idx_next = np.random.choice(
                len(neighbors),
                p=list(transition_prob.get((current_node, neighbor), 0) for neighbor in neighbors) / norm,
            )
            next_node = list(neighbors)[idx_next]
            if mode == "add":
                if (current_node, next_node) in transition_prob.keys():
                    transition_prob[(current_node, next_node)] += 1
                    print(f"Incremented edge {current_node} -> {next_node} to {transition_prob[(current_node, next_node)]}")
            tmp.append(next_node)
        walk.append(tmp)
    for i in range(5):
        print(f"Walk {i+1}: {walk[i]}")
    return walk
data_MI = pd.read_csv('./data/MI.csv', header=None, sep=';')
data_EFG = pd.read_csv('./data/EFG.csv', header=None, sep=';')
data_CW = pd.read_csv('./data/CW.csv', header=None, sep=';')
data_LFG = pd.read_csv('./data/LFG.csv', header=None, sep=';')

# drop NaN values
data_MI.dropna(how='all', inplace=True)
data_EFG.dropna(how='all', inplace=True)
data_CW.dropna(how='all', inplace=True)
data_LFG.dropna(how='all', inplace=True)

path_probabilities_CW_clustered, G_CW_clustered = path_pdf_clustered_graph(data_CW, save_paths_with_clustered_nodes = False, filename="./data/clustered_paths_CW")
path_probabilities_MI_clustered, G_MI_clustered = path_pdf_clustered_graph(data_MI, save_paths_with_clustered_nodes = False, filename="./data/clustered_paths_MI")
path_probabilities_LFG_clustered, G_LFG_clustered = path_pdf_clustered_graph(data_LFG, save_paths_with_clustered_nodes = False, filename="./data/clustered_paths_LFG")
path_probabilities_EFG_clustered, G_EFG_clustered = path_pdf_clustered_graph(data_EFG, save_paths_with_clustered_nodes = False, filename="./data/clustered_paths_EFG")

# G_MI_clustered = cluster_nodes(data_MI, num_data= 0, type_edge_weight= 'probability' )
# G_EFG_clustered = cluster_nodes(data_EFG, num_data= 0,type_edge_weight= 'probability' )
# G_CW_clustered = cluster_nodes(data_CW, num_data= 0,type_edge_weight= 'probability' )
# G_LFG_clustered = cluster_nodes(data_LFG, num_data= 0,type_edge_weight= 'probability' )

# G_CW_contracted, components_CW, contracted_nodes_CW = contract_low_degree_nodes(G_CW_clustered, position = 'flow', remove_chain =False)
# G_MI_contracted, components_MI, contracted_nodes_MI = contract_low_degree_nodes(G_MI_clustered, position = 'flow', remove_chain =False)
# G_LFG_contracted, components_LFG, contracted_nodes_LFG = contract_low_degree_nodes(G_LFG_clustered, position = 'flow', remove_chain =False)
# G_EFG_contracted, components_EFG, contracted_nodes_EFG = contract_low_degree_nodes(G_EFG_clustered, position = 'flow', remove_chain =False)

walk_CW_clustered = run_manual_random_walk(G_CW_clustered, initial_nodes, ending_nodes, iterations=1000, mode="no")
walk_MI_clustered = run_manual_random_walk(G_MI_clustered, initial_nodes, ending_nodes, iterations=1000, mode="no")
walk_LFG_clustered = run_manual_random_walk(G_LFG_clustered, initial_nodes, ending_nodes, iterations=1000, mode="no")
walk_EFG_clustered = run_manual_random_walk(G_EFG_clustered, initial_nodes, ending_nodes, iterations=1000, mode="no")

# walk_CW_contracted = run_manual_random_walk(G_CW_contracted, initial_nodes, ending_nodes, iterations=1000, mode="no")
# walk_MI_contracted = run_manual_random_walk(G_MI_contracted, initial_nodes, ending_nodes, iterations=1000, mode="no")
# walk_LFG_contracted = run_manual_random_walk(G_LFG_contracted, initial_nodes, ending_nodes, iterations=1000, mode="no")
# walk_EFG_contracted = run_manual_random_walk(G_EFG_contracted, initial_nodes, ending_nodes, iterations=1000, mode="no")

def empirical_path_distribution(data):
    empirical_paths = [' -> '.join([str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != '']) for _, row in data.iterrows()]
    counter = Counter(empirical_paths)
    total = sum(counter.values())
    distribution = {path: count / total for path, count in counter.items()}
    return distribution


empirical_path_distr_CW = empirical_path_distribution(data_CW)
empirical_path_distr_MI = empirical_path_distribution(data_MI)
empirical_path_distr_LFG = empirical_path_distribution(data_LFG)
empirical_path_distr_EFG = empirical_path_distribution(data_EFG)

def plot_path_distributions(empirical_dist, simulated_dist, top_n=20, title_suffix=''):
    all_paths = set(empirical_dist.keys()).union(simulated_dist.keys())
    ranked_paths = sorted(all_paths, key=lambda p: empirical_dist.get(p, 0), reverse=True)
    show_paths = ranked_paths[:top_n]
    emp_vals = [empirical_dist.get(p, 0) for p in show_paths]
    sim_vals = [simulated_dist.get(p, 0) for p in show_paths]

    fig = go.Figure()
    ranks = [f'{i+1}' for i in range(len(show_paths))]
    fig.add_bar(
        x=ranks,
        y=emp_vals,
        name='Empirical',
        customdata=[path for path in show_paths],
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.add_bar(
        x=ranks,
        y=sim_vals,
        name='Simulated',
        customdata=[path for path in show_paths],   
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.update_layout(
        title=f'Empirical vs Simulated Path Probability Distributions{title_suffix}',
        xaxis_title=f'Path Rank (Top {top_n})',
        yaxis_title='Probability',
        barmode='group'
    )
    fig.write_image("plots/path_random_walk.png")
    fig.show()

walk_strings_CW = [' -> '.join(w) for w in walk_CW_clustered]
walk_counter_CW = Counter(walk_strings_CW)
total_walks_CW = sum(walk_counter_CW.values())
walk_prob_CW = {walk: count / total_walks_CW for walk, count in walk_counter_CW.items()}
print("here walk CW", walk_prob_CW)

walk_strings_MI = [' -> '.join(w) for w in walk_MI_clustered]
walk_counter_MI = Counter(walk_strings_MI)
total_walks_MI = sum(walk_counter_MI.values())
walk_prob_MI = {walk: count / total_walks_MI for walk, count in walk_counter_MI.items()}    

walk_strings_LFG = [' -> '.join(w) for w in walk_LFG_clustered]
walk_counter_LFG = Counter(walk_strings_LFG)
total_walks_LFG = sum(walk_counter_LFG.values())
walk_prob_LFG = {walk: count / total_walks_LFG for walk, count in walk_counter_LFG.items()}

walk_strings_EFG = [' -> '.join(w) for w in walk_EFG_clustered]
walk_counter_EFG = Counter(walk_strings_EFG)        
total_walks_EFG = sum(walk_counter_EFG.values())
walk_prob_EFG = {walk: count / total_walks_EFG for walk, count in walk_counter_EFG.items()}

plot_path_distributions(empirical_path_distr_CW, walk_prob_CW, top_n=30, title_suffix=' (CW Clustered)')
plot_path_distributions(empirical_path_distr_MI, walk_prob_MI, top_n=30, title_suffix=' (MI Clustered)')
plot_path_distributions(empirical_path_distr_LFG, walk_prob_LFG, top_n=30, title_suffix=' (LFG Clustered)')
plot_path_distributions(empirical_path_distr_EFG, walk_prob_EFG, top_n=30, title_suffix=' (EFG Clustered)')

# walk_strings_CW_contracted = [' -> '.join(w) for w in walk_CW_contracted]
# walk_counter_CW_contracted = Counter(walk_strings_CW_contracted)
# total_walks_CW_contracted = sum(walk_counter_CW_contracted.values())
# walk_prob_CW_contracted = {walk: count / total_walks_CW_contracted for walk, count in walk_counter_CW_contracted.items()}

# walk_strings_MI_contracted = [' -> '.join(w) for w in walk_MI_contracted]
# walk_counter_MI_contracted = Counter(walk_strings_MI_contracted)
# total_walks_MI_contracted = sum(walk_counter_MI_contracted.values())
# walk_prob_MI_contracted = {walk: count / total_walks_MI_contracted for walk, count in walk_counter_MI_contracted.items()}

# walk_strings_LFG_contracted = [' -> '.join(w) for w in walk_LFG_contracted]
# walk_counter_LFG_contracted = Counter(walk_strings_LFG_contracted)
# total_walks_LFG_contracted = sum(walk_counter_LFG_contracted.values())
# walk_prob_LFG_contracted = {walk: count / total_walks_LFG_contracted for walk, count in walk_counter_LFG_contracted.items()}

# walk_strings_EFG_contracted = [' -> '.join(w) for w in walk_EFG_contracted]
# walk_counter_EFG_contracted = Counter(walk_strings_EFG_contracted)        
# total_walks_EFG_contracted = sum(walk_counter_EFG_contracted.values())
# walk_prob_EFG_contracted = {walk: count / total_walks_EFG_contracted for walk, count in walk_counter_EFG_contracted.items()}


# plot_path_distributions(empirical_path_distr_CW, walk_prob_CW_contracted, top_n=30, title_suffix=' (CW contracted)')
# plot_path_distributions(empirical_path_distr_MI, walk_prob_MI_contracted, top_n=30, title_suffix=' (MI contracted)')
# plot_path_distributions(empirical_path_distr_LFG, walk_prob_LFG_contracted, top_n=30, title_suffix=' (LFG contracted)')
# plot_path_distributions(empirical_path_distr_EFG, walk_prob_EFG_contracted, top_n=30, title_suffix=' (EFG contracted)')


In [ ]:
# path comparison with different datasets 
# CW , FBW , LFG , MI , EFG 

data_CW = pd.read_csv('./data/CW.csv', header=None, sep=';')
data_LFG = pd.read_csv('./data/LFG.csv', header=None, sep=';')
data_MI = pd.read_csv('./data/MI.csv', header=None, sep=';')
data_EFG = pd.read_csv('./data/EFG.csv', header=None, sep=';')
# data_CW = data_CW.fillna(0)  # Replace NaN with 0
# data_LFG = data_LFG.fillna(0)  # Replace NaN with 0
# data_MI = data_MI.fillna(0)  # Replace NaN with 0
# data_EFG = data_EFG.fillna(0)  # Replace NaN with 0
# data_CW = data_CW.apply(lambda x: pd.Categorical(x).codes)
# data_LFG = data_LFG.apply(lambda x: pd.Categorical(x).codes)
# data_MI = data_MI.apply(lambda x: pd.Categorical(x).codes)
# data_EFG = data_EFG.apply(lambda x: pd.Categorical(x).codes)    
# calculate the path probabilities for each dataset
path_probabilities_CW = path_pdf_weights(data_CW)
path_probabilities_LFG = path_pdf_weights(data_LFG)
path_probabilities_MI = path_pdf_weights(data_MI)
path_probabilities_EFG = path_pdf_weights(data_EFG)

# plot the path probabilities for each dataset
path_list = [path_probabilities_CW,  path_probabilities_LFG, path_probabilities_FBW, path_probabilities_MI, path_probabilities_EFG]
path_names = ["CW Dataset", "LFG Dataset", "FBW Dataset", "MI Dataset", "EFG Dataset"]
# plot the path probabilities for each dataset
path_comparison_fig_all = plot_path_abundance_multiple_comparison(
    path_list,
    names=path_names,
    show=True
)   

In [ ]:
plot_path_abundance_multiple_comparison(
    path_list[:2],
    names=path_names[:2],
    show=True
)   

# random walk analysis

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from collections import Counter

dataset_path = "./data/clustered_paths_CW.csv"  # Change to any of your clusters as needed
df = pd.read_csv(dataset_path, sep=';', header=None)

G = nx.DiGraph()
transition_counts = Counter()
initial_counts = Counter()
ending_counts = Counter()
num_pots = df.shape[0]

for idx, row in df.iterrows():
    steps = [str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != '']
    print(steps)
    if not steps:
        continue
    initial_counts[steps[0]] += 1
    ending_counts[steps[-1]] += 1
    for i in range(len(steps)-1):
        transition_counts[(steps[i], steps[i+1])] += 1
        G.add_edge(steps[i], steps[i+1])

for (u, v), w in transition_counts.items():
    G[u][v]['weight'] = w
# print the weights of the edges
for u, v, data in G.edges(data=True):
    print(f"Edge from {u} to {v} has weight {data['weight']}")
# Transition probabilities from transition counts
transition_prob = {}
for u in G.nodes():
    outgoing = [v for v in G.successors(u)]
    total = sum([transition_counts[(u, v)] for v in outgoing])
    for v in outgoing:
        prob = transition_counts[(u,v)] / total if total > 0 else 0
        transition_prob[(u,v)] = prob
initial_nodes = {node: c/num_pots for node, c in initial_counts.items()}
ending_nodes = {node: c/num_pots for node, c in ending_counts.items()}

def simulate_random_walks(G, initial_nodes, ending_nodes, transition_prob, n_walks=1000):
    walks = []
    zero_indegree_nodes = [node for node, degree in G.in_degree() if degree == 0]
    p_init = np.array([initial_nodes.get(node, 0) for node in zero_indegree_nodes])
    if p_init.sum() > 0:
        p_init = p_init / p_init.sum()
    else:
        p_init = np.full(len(zero_indegree_nodes), 1 / len(zero_indegree_nodes))
    for i in range(n_walks):
        walk = []
        idx = np.random.choice(len(zero_indegree_nodes), p=p_init)
        current_node = zero_indegree_nodes[idx]
        walk.append(current_node)
        while G.out_degree(current_node) != 0:
            neighbors = list(G.successors(current_node))
            probs = np.array([transition_prob.get((current_node, neighbor), 0) for neighbor in neighbors])
            if probs.sum() > 0:
                probs = probs / probs.sum()
            else:
                probs = np.full(len(neighbors), 1 / len(neighbors))
            idx_next = np.random.choice(len(neighbors), p=probs)
            next_node = neighbors[idx_next]
            walk.append(next_node)
            if next_node in ending_nodes:
                if np.random.rand() < ending_nodes[next_node]:
                    break
            current_node = next_node
        walks.append(walk)
    walk_strings = [' -> '.join(w) for w in walks]
    walk_counter = Counter(walk_strings)
    total_walks = sum(walk_counter.values())
    walk_prob = {walk: count / total_walks for walk, count in walk_counter.items()}
    return walk_prob


simulated_path_distribution = simulate_random_walks(G, initial_nodes, ending_nodes, transition_prob, n_walks=10000)

# --- Compare empirical path distribution ---
empirical_paths = [' -> '.join([str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != '']) for _, row in df.iterrows()]
empirical_counter = Counter(empirical_paths)
empirical_total = sum(empirical_counter.values())
empirical_path_distribution = {path: count / empirical_total for path, count in empirical_counter.items()}

# --- Print example comparison of most common paths ---
print("Most common empirical paths:")
for path, prob in sorted(empirical_path_distribution.items(), key=lambda x: -x[1])[:10]:
    print(f'{path}: {prob*100:.2f}%')

print("\nMost common simulated paths:")
for path, prob in sorted(simulated_path_distribution.items(), key=lambda x: -x[1])[:10]:
    print(f'{path}: {prob*100:.2f}%')


In [ ]:
import matplotlib.pyplot as plt

# def plot_path_distributions(empirical_dist, simulated_dist, top_n=20):
#     # Get all unique paths
#     all_paths = set(empirical_dist.keys()).union(simulated_dist.keys())
#     # Rank paths by empirical frequency
#     ranked_paths = sorted(all_paths, key=lambda p: empirical_dist.get(p, 0), reverse=True)
#     show_paths = ranked_paths[:top_n]
#     # Prepare values
#     emp_vals = [empirical_dist.get(p, 0) for p in show_paths]
#     sim_vals = [simulated_dist.get(p, 0) for p in show_paths]
    
#     x = range(len(show_paths))
#     plt.figure(figsize=(16,7))
#     plt.bar(x, emp_vals, width=0.4, label='Empirical', align='center', color='C0')
#     plt.bar([i+0.4 for i in x], sim_vals, width=0.4, label='Simulated', align='center', color='C1')
#     plt.xticks([i+0.2 for i in x], [f'{i+1}' for i in x], rotation=90)
#     plt.ylabel('Probability')
#     plt.xlabel('Path Rank (Top {})'.format(top_n))
#     plt.title('Empirical vs Simulated Path Probability Distributions')
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

#     # Optionally, print which paths correspond to which rank
#     for idx, path in enumerate(show_paths):
#         print(f'Rank {idx+1}: {path[:120]}...')  # Show start of path, truncate for readability
import plotly.graph_objects as go

def plot_path_distributions(empirical_dist, simulated_dist, top_n=20):
    all_paths = set(empirical_dist.keys()).union(simulated_dist.keys())
    ranked_paths = sorted(all_paths, key=lambda p: empirical_dist.get(p, 0), reverse=True)
    show_paths = ranked_paths[:top_n]
    emp_vals = [empirical_dist.get(p, 0) for p in show_paths]
    sim_vals = [simulated_dist.get(p, 0) for p in show_paths]

    fig = go.Figure()
    ranks = [f'{i+1}' for i in range(len(show_paths))]
    fig.add_bar(
        x=ranks,
        y=emp_vals,
        name='Empirical',
        customdata=[path for path in show_paths],
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.add_bar(
        x=ranks,
        y=sim_vals,
        name='Simulated',
        customdata=[path for path in show_paths],
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.update_layout(
        title='Empirical vs Simulated Path Probability Distributions',
        xaxis_title=f'Path Rank (Top {top_n})',
        yaxis_title='Probability',
        barmode='group'
    )
    fig.write_image("plots/path_random_walk.png")
    fig.show()

    for idx, path in enumerate(show_paths):
        print(f'Rank {idx+1}: {path[:120]}...')
# Usage: after the main code above
walk_prob = {walk: count / 1000 for walk, count in walk_counter.items()}
plot_path_distributions(empirical_path_distribution, simulated_path_distribution, top_n=30)


In [ ]:
all_paths = set(empirical_path_distribution.keys()).union(simulated_path_distribution.keys())
ranked_paths = sorted(all_paths, key=lambda p: empirical_path_distribution.get(p, 0), reverse=True)
show_paths = ranked_paths[:50]
print("Top 20 paths:")
for path in show_paths:
    emp_prob = empirical_path_distribution.get(path, 0)
    sim_prob = simulated_path_distribution.get(path, 0)
    print(f'Path: {path}\n  Empirical Probability: {emp_prob*100:.2f}%\n  Simulated Probability: {sim_prob*100:.2f}%\n')

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd

dataset_files = {
    'CW': "./data/clustered_paths_CW.csv",
    'LFG': "./data/clustered_paths_LFG.csv",
    'MI': "./data/clustered_paths_MI.csv",
    'EFG': "./data/clustered_paths_EFG.csv"
}

all_empirical_distributions = {}

for name, path in dataset_files.items():
    df = pd.read_csv(path, sep=';', header=None)
    empirical_paths = [' -> '.join([str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != '']) for _, row in df.iterrows()]
    counter = Counter(empirical_paths)
    total = sum(counter.values())
    distribution = {path: count / total for path, count in counter.items()}
    all_empirical_distributions[name] = distribution

# Rank all paths by overall frequency (can also rank within individual societies)
all_unique_paths = set()
for d in all_empirical_distributions.values():
    all_unique_paths |= set(d.keys())

ranked_paths = sorted(all_unique_paths, key=lambda p: sum(d.get(p,0) for d in all_empirical_distributions.values()), reverse=True)
top_n = 20
show_paths = ranked_paths[:top_n]

x = range(len(show_paths))
plt.figure(figsize=(18,7))
plt.grid(axis='y', linestyle='--', alpha=0.6)
colors = ['C0','C1','C2','C3']
for i, (name, dist) in enumerate(all_empirical_distributions.items()):
    vals = [dist.get(p, 0) for p in show_paths]
    plt.bar([xi+i*0.2 for xi in x], vals, width=0.2, label=name, align='center', color=colors[i])
plt.xticks([xi+0.3 for xi in x], [f'{i+1}' for i in x], rotation=90)
plt.ylabel('Probability')
plt.xlabel('Path Rank (Top {})'.format(top_n))
plt.title('Empirical Path Distributions Across Societies')
plt.legend()
plt.tight_layout()

plt.savefig("plots/path_empirical_comparison.pdf")
plt.show()

# Print which sequence corresponds to each rank
for idx, path in enumerate(show_paths):
    print(f'Rank {idx+1}: {path[:120]}...')


In [ ]:
import ast
from itertools import chain, combinations

dataset_files = {
    'CW': "./data/clustered_paths_CW.csv",
    'LFG': "./data/clustered_paths_LFG.csv",
    'MI': "./data/clustered_paths_MI.csv",
    'EFG': "./data/clustered_paths_EFG.csv",
    # 'CW': "./data/bootstrap_documents_CW.csv",
    # 'LFG': "./data/bootstrap_documents_LFG.csv",
    # 'MI': "./data/bootstrap_documents_MI.csv",
    # 'EFG': "./data/bootstrap_documents_EFG.csv",

}

def parse_step_cell(cell):
    if pd.isna(cell):
        return None
    text = str(cell).strip().strip('"').strip("'")
    if not text:
        return None
    if text.startswith("(") and text.endswith(")"):
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return None
        if isinstance(parsed, tuple):
            return tuple(part.strip().casefold() for part in parsed)
    return None  # ignore non-triplet tokens such as IDs

import re

def load_sequences(path):
    df = pd.read_csv(path, sep=';', header=None, dtype=str)
    sequences = []
    for _, row in df.iterrows():
        triples = []
        row_text = " ".join(str(cell) for cell in row.dropna() if str(cell).strip())
        for match in re.findall(r"\([^()]+\)", row_text):
            parsed = parse_step_cell(match)
            if parsed is not None:
                triples.append(parsed)
        if triples:
            sequences.append(tuple(triples))
    return sequences

path_sets, technique_sets = {}, {}
for name, path in dataset_files.items():
    sequences = load_sequences(path)
    path_sets[name] = set(sequences)
    technique_sets[name] = set(chain.from_iterable(sequences))
    print(f"{name}: {len(sequences)} cleaned paths")

shared_techniques_all = set.intersection(*technique_sets.values())
shared_paths_all = set.intersection(*path_sets.values())

print(f"\nShared techniques ({len(shared_techniques_all)}):")
for tech in sorted(shared_techniques_all):
    print(tech)

print(f"\nShared paths ({len(shared_paths_all)}):")
for path in sorted(shared_paths_all):
    print(path)

def pairwise_overlaps(sets):
    pairs = list(combinations(sets.keys(), 2))
    return {pair: sets[pair[0]] & sets[pair[1]] for pair in pairs}

tech_overlaps = pairwise_overlaps(technique_sets)
path_overlaps = pairwise_overlaps(path_sets)

In [ ]:
def transfer_entropy_matrix(sequences_a, sequences_b):
    counter_a = Counter()
    counter_b = Counter()
    marginal_a = Counter()
    marginal_b = Counter()
    for seq in sequences_a:
        for i in range(len(seq)-1):
            counter_a[(seq[i], seq[i+1])] += 1
            marginal_a[seq[i]] += 1
    for seq in sequences_b:
        for i in range(len(seq)-1):
            counter_b[(seq[i], seq[i+1])] += 1
            marginal_b[seq[i]] += 1
    te_sum = 0
    print(marginal_a)
    for (x0, x1) in set(counter_a.keys()).intersection(set(counter_b.keys())):
        p_a = counter_a[(x0, x1)] / marginal_a[x0] if marginal_a[x0] > 0 else 0
        p_b = counter_b[(x0, x1)] / marginal_b[x0] if marginal_b[x0] > 0 else 0
        p_joint = (counter_a[(x0, x1)] + counter_b[(x0, x1)]) / (marginal_a[x0] + marginal_b[x0])
        if p_a > 0 and p_b > 0 and p_joint > 0:
            te_sum += p_joint * np.log2(p_joint/(p_a*p_b))
    return te_sum

societies = list(dataset_files.keys())
te_matrix = np.zeros((len(societies), len(societies)))
sequences = {}
# for name, path in dataset_files.items():
#     df = pd.read_csv(path, sep=';', header=None)
#     sequences[name] = [[str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != ''] for _, row in df.iterrows()]
sequences = {name: load_sequences(path) for name, path in dataset_files.items()}
print(sequences.keys())
print(sequences.values())  # Print first two sequences for CW to verify parsing
for i, s1 in enumerate(societies):
    for j, s2 in enumerate(societies):
        print(sequences[s1])
        if i < j:
            te_matrix[i, j] = transfer_entropy_matrix(sequences[s1], sequences[s2])
        elif i > j:
            te_matrix[i, j] = transfer_entropy_matrix(sequences[s2], sequences[s1])
        else:
            te_matrix[i, j] = 0  # Not defined for self

print("\nTransfer Entropy Matrix (upper triangle):")
print(te_matrix)


In [ ]:
plt.figure(figsize=(7,6))
import seaborn as sns
sns.heatmap(te_matrix, annot=True, cmap='magma', xticklabels=societies, yticklabels=societies, fmt='.2f')
plt.title('Pairwise Transfer Entropy Between Societies')
plt.show()


In [ ]:
import ast
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import networkx as nx
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors


def _clean_token(raw: str) -> str:
    token = str(raw).strip().strip('"').strip("'")
    return token if token else ""


def _parse_cell(cell: str) -> Optional[str]:
    token = _clean_token(cell)
    if not token:
        return None
    if token.startswith("(") and token.endswith(")"):
        try:
            parsed = ast.literal_eval(token)
            if isinstance(parsed, (tuple, list)):
                return "⟨" + "|".join(str(part).strip().lower() for part in parsed) + "⟩"
        except (ValueError, SyntaxError):
            return None
    return token.lower()


def load_sequences_csv(csv_path: Path, sep: str = ";") -> List[List[str]]:
    df = pd.read_csv(csv_path, sep=sep, header=None, dtype=str)
    sequences: List[List[str]] = []
    for _, row in df.iterrows():
        seq = [token for cell in row.dropna() if (token := _parse_cell(cell))]
        if seq:
            sequences.append(seq)
    return sequences


def build_markov_components(
    sequences: Sequence[Sequence[str]],
) -> Tuple[nx.DiGraph, Dict[str, float], Dict[str, float], Dict[Tuple[str, str], float]]:
    G = nx.DiGraph()
    transition_counts: Counter = Counter()
    initial_counts: Counter = Counter()
    terminal_counts: Counter = Counter()

    for seq in sequences:
        if not seq:
            continue
        initial_counts[seq[0]] += 1
        terminal_counts[seq[-1]] += 1
        for u, v in zip(seq[:-1], seq[1:]):
            transition_counts[(u, v)] += 1
            G.add_edge(u, v)

    for (u, v), w in transition_counts.items():
        G[u][v]["weight"] = w

    transition_prob: Dict[Tuple[str, str], float] = {}
    for u in G.nodes:
        successors = list(G.successors(u))
        if not successors:
            continue
        total = sum(transition_counts[(u, v)] for v in successors)
        for v in successors:
            transition_prob[(u, v)] = transition_counts[(u, v)] / total if total > 0 else 0.0

    total_sequences = sum(initial_counts.values()) or 1
    initial_prob = {node: count / total_sequences for node, count in initial_counts.items()}
    terminal_prob = {node: count / total_sequences for node, count in terminal_counts.items()}
    return G, initial_prob, terminal_prob, transition_prob


def simulate_random_walks(
    graph: nx.DiGraph,
    initial_prob: Dict[str, float],
    terminal_prob: Dict[str, float],
    transition_prob: Dict[Tuple[str, str], float],
    *,
    n_walks: int = 5000,
    max_steps: int = 25,
    random_state: Optional[int] = None,
) -> List[List[str]]:
    rng = np.random.default_rng(random_state)
    starts = [node for node, indeg in graph.in_degree() if indeg == 0] or list(graph.nodes)
    start_weights = np.array([initial_prob.get(node, 0.0) for node in starts])
    if start_weights.sum() == 0:
        start_weights = np.full(len(starts), 1 / len(starts))
    else:
        start_weights = start_weights / start_weights.sum()

    walks: List[List[str]] = []
    for _ in range(n_walks):
        current = rng.choice(starts, p=start_weights)
        walk = [current]
        steps = 0
        while steps < max_steps and graph.out_degree(current) > 0:
            neighbors = list(graph.successors(current))
            probs = np.array([transition_prob.get((current, nbr), 0.0) for nbr in neighbors])
            if probs.sum() == 0:
                probs = np.full(len(neighbors), 1 / len(neighbors))
            else:
                probs /= probs.sum()
            current = rng.choice(neighbors, p=probs)
            walk.append(current)
            steps += 1
            if rng.random() < terminal_prob.get(current, 0.0):
                break
        walks.append(walk)
        print(f"Generated walk of length {walk}")
    return walks


def train_path_embedding(
    walks_by_culture: Dict[str, List[List[str]]],
    *,
    vector_size: int = 64,
    window: int = 3,
    min_count: int = 1,
    epochs: int = 30,
    sg: int = 1,
    random_state: Optional[int] = None,
) -> Word2Vec:
    all_walks = list(walk for walks in walks_by_culture.values() for walk in walks)
    model = Word2Vec(
        sentences=all_walks,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=sg,
        epochs=epochs,
        seed=random_state,
        workers=1,
    )
    return model


def embed_paths(
    walks_by_culture: Dict[str, List[List[str]]],
    model: Word2Vec,
) -> Tuple[np.ndarray, List[str], List[str]]:
    embeddings: List[np.ndarray] = []
    labels: List[str] = []
    identifiers: List[str] = []
    for culture, walks in walks_by_culture.items():
        for idx, walk in enumerate(walks):
            tokens = [token for token in walk if token in model.wv]
            if not tokens:
                continue
            vectors = model.wv[tokens]
            embeddings.append(vectors.mean(axis=0))
            labels.append(culture)
            identifiers.append(f"{culture}::walk_{idx}")
    return np.vstack(embeddings), labels, identifiers


def compute_culture_statistics(
    embeddings: np.ndarray,
    labels: Sequence[str],
) -> Dict[str, Dict[str, np.ndarray]]:
    stats: Dict[str, Dict[str, np.ndarray]] = {}
    for culture in sorted(set(labels)):
        idx = [i for i, lbl in enumerate(labels) if lbl == culture]
        culture_emb = embeddings[idx]
        centroid = culture_emb.mean(axis=0)
        cov = np.cov(culture_emb, rowvar=False) if culture_emb.shape[0] > 1 else np.eye(culture_emb.shape[1])
        stats[culture] = {
            "centroid": centroid,
            "covariance": cov,
            "mean_distance": np.linalg.norm(culture_emb - centroid, axis=1).mean(),
        }
    return stats


def pairwise_centroid_distances(stats: Dict[str, Dict[str, np.ndarray]]) -> pd.DataFrame:
    cultures = sorted(stats.keys())
    centroids = np.vstack([stats[c]["centroid"] for c in cultures])
    dist_matrix = pairwise_distances(centroids, metric="euclidean")
    return pd.DataFrame(dist_matrix, index=cultures, columns=cultures)


def nearest_neighbors_by_culture(
    embeddings: np.ndarray,
    labels: Sequence[str],
    identifiers: Sequence[str],
    *,
    k: int = 5,
) -> pd.DataFrame:
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(embeddings)), metric="euclidean")
    nn.fit(embeddings)
    distances, indices = nn.kneighbors(embeddings)
    rows = []
    for anchor_idx, (dists, nbr_idx) in enumerate(zip(distances, indices)):
        for dist, idx in zip(dists[1:], nbr_idx[1:]):
            rows.append(
                {
                    "source_id": identifiers[anchor_idx],
                    "source_culture": labels[anchor_idx],
                    "neighbor_id": identifiers[idx],
                    "neighbor_culture": labels[idx],
                    "distance": dist,
                }
            )
    return pd.DataFrame(rows)


def flag_outliers(
    embeddings: np.ndarray,
    labels: Sequence[str],
    identifiers: Sequence[str],
    *,
    z_threshold: float = 2.5,
) -> pd.DataFrame:
    rows = []
    for culture in sorted(set(labels)):
        idx = [i for i, lbl in enumerate(labels) if lbl == culture]
        culture_emb = embeddings[idx]
        centroid = culture_emb.mean(axis=0)
        dists = np.linalg.norm(culture_emb - centroid, axis=1)
        mean = dists.mean()
        std = dists.std() or 1.0
        z_scores = (dists - mean) / std
        for local_idx, z in enumerate(z_scores):
            if z >= z_threshold:
                global_idx = idx[local_idx]
                rows.append(
                    {
                        "path_id": identifiers[global_idx],
                        "culture": culture,
                        "distance": dists[local_idx],
                        "z_score": z,
                    }
                )
    return pd.DataFrame(rows)


def temporal_trajectory(
    embeddings: np.ndarray,
    labels: Sequence[str],
    identifiers: Sequence[str],
    chronology: Dict[str, float],
) -> pd.DataFrame:
    records = []
    for emb, lbl, pid in zip(embeddings, labels, identifiers):
        if pid not in chronology:
            continue
        records.append({"path_id": pid, "culture": lbl, "time": chronology[pid], "embedding": emb})
    if not records:
        return pd.DataFrame(columns=["path_id", "culture", "time", "embedding"])
    df = pd.DataFrame(records)
    return df.sort_values("time").reset_index(drop=True)


def run_random_walk_embedding_analysis(
    dataset_map: Dict[str, Path],
    *,
    n_walks: int = 5000,
    max_steps: int = 25,
    vector_size: int = 64,
    chronology: Optional[Dict[str, float]] = None,
    random_state: Optional[int] = 42,
) -> Dict[str, object]:
    sequences_per_culture = {culture: load_sequences_csv(path) for culture, path in dataset_map.items()}
    walks_by_culture: Dict[str, List[List[str]]] = {}
    for culture, sequences in sequences_per_culture.items():
        graph, init_p, term_p, trans_p = build_markov_components(sequences)
        walks_by_culture[culture] = simulate_random_walks(
            graph,
            init_p,
            term_p,
            trans_p,
            n_walks=n_walks,
            max_steps=max_steps,
            random_state=random_state,
        )

    model = train_path_embedding(
        walks_by_culture,
        vector_size=vector_size,
        random_state=random_state,
    )
    embeddings, labels, identifiers = embed_paths(walks_by_culture, model)
    stats = compute_culture_statistics(embeddings, labels)
    centroid_distances = pairwise_centroid_distances(stats)
    neighbors = nearest_neighbors_by_culture(embeddings, labels, identifiers)
    outliers = flag_outliers(embeddings, labels, identifiers)
    temporal = temporal_trajectory(embeddings, labels, identifiers, chronology or {})

    return {
        "model": model,
        "embeddings": embeddings,
        "labels": labels,
        "identifiers": identifiers,
        "statistics": stats,
        "centroid_distances": centroid_distances,
        "nearest_neighbors": neighbors,
        "outliers": outliers,
        "temporal": temporal,
    }
from pathlib import Path

datasets = {
    "CW": Path("./data/clustered_paths_CW.csv"),
    "LFG": Path("./data/clustered_paths_LFG.csv"),
    "MI": Path("./data/clustered_paths_MI.csv"),
    "EFG": Path("./data/clustered_paths_EFG.csv"),
}

result = run_random_walk_embedding_analysis(
    datasets,
    n_walks=10000,
    max_steps=20,
    vector_size=64,
    random_state=123,
)

embeddings = result["embeddings"]
centroid_distances = result["centroid_distances"]
nearest_neighbors = result["nearest_neighbors"]
outliers = result["outliers"]
print(centroid_distances)
print(nearest_neighbors.head())
print(outliers.head())


In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE

tsne = TSNE(n_components=3, perplexity=50, random_state=123)
embedding_2d = tsne.fit_transform(embeddings)

fig_tsne = px.scatter(
    x=embedding_2d[:, 0],
    y=embedding_2d[:, 1],
    color=result["labels"],
    hover_name=result["identifiers"],
    title="t-SNE of random-walk embeddings",
)
fig_tsne.show()

fig_heat = px.imshow(
    result["centroid_distances"],
    labels=dict(x="Culture", y="Culture", color="Distance"),
    title="Centroid distance matrix",
    text_auto=".2f",
)
fig_heat.show()

nn_counts = (
    result["nearest_neighbors"]
    .groupby(["source_culture", "neighbor_culture"])
    .size()
    .reset_index(name="count")
)

fig_nn = px.sunburst(
    nn_counts,
    path=["source_culture", "neighbor_culture"],
    values="count",
    title="Nearest-neighbour cross-culture links",
)
fig_nn.show()

#do the same with umap 

In [ ]:
fig_tsne = px.scatter_3d(
    x=embedding_2d[:, 0],
    y=embedding_2d[:, 1],
    z=embedding_2d[:, 2],
    color=result["labels"],
    hover_name=result["identifiers"],
    title="t-SNE of random-walk embeddings",
)
fig_tsne.show()

# random walks on graphs

In [ ]:
# find the ending nodes in the FBW clustered dataset
data_FBW_clustered = pd.read_csv('./data/clustered_paths_FBW.csv', header=None, sep=';')
# length of the clustered paths
print("Number of clustered paths in FBW dataset:", len(data_FBW_clustered))
# for each row do not consider NaN values
data_FBW_clustered.dropna()
data_FBW_clustered.head()  
# ending nodes a dictionary with the keys being the ending nodes and the values being the count of occurrences
ending_nodes = dict()
for index, row in data_FBW_clustered.iterrows():
    # Get the last non-NaN value in the row
    last_value = row.dropna().values[-1]
    ending_nodes[last_value] = ending_nodes.get(last_value, 0) + 1
print("Ending nodes in the FBW clustered dataset:")
for node, count in ending_nodes.items():
    # adjust the values probability by the number of paths
    ending_nodes[node] = count / len(data_FBW_clustered)
    print("node", node, "count", count, "probability", ending_nodes[node])

for node, prob in ending_nodes.items():
    print("node", node, "probability", prob)

initial_nodes = dict()
for index, row in data_FBW_clustered.iterrows():
    # Get the first non-NaN value in the row
    first_value = row.dropna().values[0]
    initial_nodes[first_value] = initial_nodes.get(first_value, 0) + 1
print("Initial nodes in the FBW clustered dataset:")
for node, count in initial_nodes.items():
    # adjust the values probability by the number of paths
    initial_nodes[node] = count / len(data_FBW_clustered)
    print("node", node, "count", count, "probability", initial_nodes[node])

In [ ]:
#gv.d3(G_clustered, show_edge_label=True, edge_label_data_source = 'label')
gv.d3(G_contracted, **graph_options)

In [ ]:
# Simulation of random walk in the graph from a starting node to an end node the end node is not specified because the network is acyclic and the walk stops when no outgoing edges are available
import random
G_CW_clustered = G_MI_clustered
mode = 'no'
def _edge_key(u, v):
    return (u, v)
def init_edge_weights(G: nx.DiGraph, weight_attr='edge_counting', default=1.0):
    base_w = {}
    for u, v, d in G.edges(data=True):
        base_w[(u, v)] = float(d.get(weight_attr, default))
        print(base_w)
    return base_w
iterations = 1000
transition_prob = nx.get_edge_attributes(G_CW_clustered, 'edge_counting')
for (u, v), weight in transition_prob.items():
    print(f"Edge: {u} -> {v}, Initial weight: {weight}")
walk = []
for _ in range(iterations):
    tmp = []
    zero_indegree_nodes = [node for node, degree in G_CW_clustered.in_degree() if degree == 0]
    idx = np.random.choice(len(zero_indegree_nodes), p=list(initial_nodes.get(node, 0) for node in zero_indegree_nodes)/np.sum(list(initial_nodes.get(node, 0) for node in zero_indegree_nodes)))
    tmp.append(zero_indegree_nodes[idx])
    while G_CW_clustered.out_degree(tmp[-1]) != 0:
        current_node = tmp[-1]
        if current_node not in G_CW_clustered:
            break
        if current_node in ending_nodes.keys() and random.random() > ending_nodes[current_node]:
            break
        neighbors = list(G_CW_clustered.successors(current_node))
        print(f"Current node: {current_node}, Neighbors: {neighbors}")
        # sum of transition probabilities to normalize
        norm = np.sum(list(transition_prob[(current_node, neighbor)] for neighbor in neighbors))
        idx_next = np.random.choice(len(neighbors), p=list(transition_prob.get((current_node, neighbor), 0) for neighbor in neighbors)/norm)
        next_node = list(neighbors)[idx_next]
        if mode == 'add':
            # increment the transition prob by one counting based on the walk
            if (current_node, next_node) in transition_prob.keys():
                transition_prob[(current_node, next_node)] += 1
                print(f"Incremented edge {current_node} -> {next_node} to {transition_prob[(current_node, next_node)]}")
        tmp.append(next_node)
        # init_edge_weights(G_FBW)
    walk.append(tmp)
# print the first 5 walks
for i in range(5):
    print(f"Walk {i+1}: {walk[i]}")


In [ ]:
def plot_path_distributions(empirical_dist, simulated_dist, top_n=20):
    all_paths = set(empirical_dist.keys()).union(simulated_dist.keys())
    ranked_paths = sorted(all_paths, key=lambda p: empirical_dist.get(p, 0), reverse=True)
    show_paths = ranked_paths[:top_n]
    emp_vals = [empirical_dist.get(p, 0) for p in show_paths]
    sim_vals = [simulated_dist.get(p, 0) for p in show_paths]

    fig = go.Figure()
    ranks = [f'{i+1}' for i in range(len(show_paths))]
    fig.add_bar(
        x=ranks,
        y=emp_vals,
        name='Empirical',
        customdata=[path for path in show_paths],
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.add_bar(
        x=ranks,
        y=sim_vals,
        name='Simulated',
        customdata=[path for path in show_paths],
        hovertemplate="Rank %{x}<br>Probability %{y:.4f}<br><b>%{customdata}</b><extra></extra>"
    )
    fig.update_layout(
        title='Empirical vs Simulated Path Probability Distributions',
        xaxis_title=f'Path Rank (Top {top_n})',
        yaxis_title='Probability',
        barmode='group'
    )
    fig.write_image("plots/path_random_walk.png")
    fig.show()

    for idx, path in enumerate(show_paths):
        print(f'Rank {idx+1}: {path[:120]}...')
# Usage: after the main code above
walk_strings = [' -> '.join(w) for w in walk]
walk_counter = Counter(walk_strings)
total_walks = sum(walk_counter.values())
walk_prob = {walk: count / total_walks for walk, count in walk_counter.items()}
data_MI_clustered = pd.read_csv('./data/clustered_paths_MI.csv', header=None, sep=';')
empirical_paths = [' -> '.join([str(step).strip() for step in row if pd.notnull(step) and str(step).strip() != '']) for _, row in data_MI_clustered.iterrows()]
empirical_counter = Counter(empirical_paths)
empirical_total = sum(empirical_counter.values())
empirical_path_distr = {path: count / empirical_total for path, count in empirical_counter.items()}
plot_path_distributions(empirical_path_distr, walk_prob, top_n=30)

In [ ]:
all_paths = set(empirical_path_distr_MI.keys()).union(walk_prob.keys())
ranked_paths = sorted(all_paths, key=lambda p: empirical_path_distr_MI.get(p, 0), reverse=True)
print("Top 20 paths:")
for path in ranked_paths[:20]:
    emp_prob = empirical_path_distr_MI.get(path, 0)
    sim_prob = walk_prob.get(path, 0)
    print(f'Path: {path}\n  Empirical Probability: {emp_prob*100:.2f}%\n  Simulated Probability: {sim_prob*100:.2f}%\n')

# III. Data analysis

In this section we focus on finding an appropriate way to explain the influence or the contaminations of one dataset to another. Furthermore, we explore the possibility to find a null model that can explain the path formation.

For the sake of clarity this section will be divided in: 
* Some centrality measures to find "most infomatives nodes"
* Null models
* Transfer Entropy analysis
* Bayesian attempt

### III.1 Centrality measure attempts

There are a lot of centrality measures, few for directed networks. 

* The **current-flow betweenness centrality** $b_i$ of node $i$ is the flow through $i$ averaged over all source-target pairs $(s,t)$:

$$\displaystyle{b_i = \frac{\sum_{s < t} F_{i}^{(s,t)}}{(1/2) n (n-1)}}$$

However is still not defined for directed networks. 

* The **Pagerank** is defined for directed networks, but in this case is a bit biased by the initial and final nodes

In [ ]:
data_FBW = pd.read_csv('./data/FBW.csv', header=None, sep=';')
# unique paths in the dataset delete the duplicate rows
# Note: The original code had a syntax error in the iloc method, it should be
unique_paths = data_FBW.iloc[:, :].drop_duplicates().shape[0]
print(f"Number of unique nodes in the FBW dataset: {unique_paths}")
G_FBW = graph_from_data(data_FBW)
c_f_b_c = nx.current_flow_betweenness_centrality(G_FBW.to_undirected(), weight='weight')
# print the top 10 nodes with highest current flow betweenness centrality
top_10_c_f_b_c = sorted(c_f_b_c.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 nodes with highest current flow betweenness centrality:")
for node, centrality in top_10_c_f_b_c:
    print(f"Node {node}: {centrality:.4f}")

In [ ]:
data_FBW = pd.read_csv('./data/FBW.csv', header=None, sep=';')
G_FBW = graph_from_data(data_FBW)
c_f_b_c = nx.pagerank(G_FBW, weight='weight')
# print the top 10 nodes with highest current flow betweenness centrality
top_10_c_f_b_c = sorted(c_f_b_c.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 nodes with highest current flow betweenness centrality:")
for node, centrality in top_10_c_f_b_c:
    print(f"Node {node}: {centrality:.4f}")

In [ ]:
data_FBW = pd.read_csv('./data/FBW.csv', header=None, sep=';')
G_FBW = cluster_nodes(data_FBW, num_data=0, coloring='cool', type_edge_weight = 'probability')

c_f_b_c = nx.current_flow_betweenness_centrality(G_FBW.to_undirected(), weight='weight')
# print the top 10 nodes with highest current flow betweenness centrality
top_10_c_f_b_c = sorted(c_f_b_c.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 nodes with highest current flow betweenness centrality:")
for node, centrality in top_10_c_f_b_c:
    print(f"Node {node}: {centrality:.4f}")

In [ ]:
data_CW = pd.read_csv('./data/CW.csv', header=None, sep=';')
G_CW = cluster_nodes(data_CW, num_data=0, coloring='cool', type_edge_weight = 'probability')
c_f_b_c = nx.current_flow_betweenness_centrality(G_CW.to_undirected(), weight='weight')
# print the top 10 nodes with highest current flow betweenness centrality
top_10_c_f_b_c = sorted(c_f_b_c.items(), key=lambda x: x[1], reverse=True)[:10]
print("\nTop 10 nodes with highest current flow betweenness centrality:")
for node, centrality in top_10_c_f_b_c:
    print(f"Node {node}: {centrality:.4f}")

# III.2 Uncertanty in data (Bootstrapping)

The main idea in this section is the following: resampling techniques, then compute some quantities, look at robust network estimations, those can be used for a maximum entropy approach. 

This is because archaeological data usually are corrupted or not complete. Therefore, if in the small dataset we find some quantities that are robust under missing nodes or edges is likely that the same characteristic or quantity is robust in general. Hence, that quantity is more suitable as archeological network constraint to be used in the maximum entropy machinery. 

In [ ]:
def resample_from_dataset(data, sample_fraction=0.5):
    """
    Resample paths from the dataset with replacement.
    
    Parameters:
    - data: DataFrame containing the paths.
    - sample_fraction: Fraction of paths to sample.
    
    Returns:
    - bootstrapped_data: DataFrame with resampled paths.
    """
    n_samples = int(len(data) * sample_fraction)
    bootstrapped_data = data.sample(n=n_samples, replace=True)
    return bootstrapped_data.reset_index(drop=True)

# ok we need centrality measures for the clustered graph
def calculate_centrality_measures(G):
    """
    Calculate various centrality measures for the given graph.
    
    Parameters:
    - G: NetworkX graph object.
    
    Returns:
    - centrality: Dictionary containing centrality measures.
    """
    centrality = {}
    try:
        G_undir = G.to_undirected()
        centrality['degree'] = dict(G.degree())
        centrality['betweenness'] = nx.betweenness_centrality(G_undir, weight='weight')
        centrality['pagerank'] = nx.pagerank(G_undir, weight='weight')
        centrality['current_flow'] = nx.current_flow_betweenness_centrality(G_undir, weight='weight')
        centrality['eigenvector'] = nx.eigenvector_centrality(G_undir, weight='weight')
        centrality['closeness'] = nx.closeness_centrality(G_undir, distance='weight')
        centrality['katz'] = nx.katz_centrality_numpy(G_undir, weight='weight')
    except:
        centrality['current_flow'] = {node: 0 for node in G.nodes()}
        centrality['eigenvector'] = {node: 0 for node in G.nodes()}
        centrality['closeness'] = {node: 0 for node in G.nodes()}
        centrality['katz'] = {node: 0 for node in G.nodes()}
    return centrality

G_clustered = cluster_nodes(data_FBW, num_data = 0, coloring= 'cool')
clustered_paths_FBW = pd.read_csv('./data/clustered_paths_FBW.csv', header=None, sep=';')
G_resampled = resample_from_dataset(clustered_paths_FBW, sample_fraction=0.5)
gv.d3(G_clustered, **graph_options)



In [ ]:
import networkx as nx

# create a graph erdosh reny 

G = nx.erdos_renyi_graph(10, 0.3)
A = nx.adjacency_matrix(G)
print("Adjacency matrix of the graph:")
print(A.todense())
# diagonalase the adjacency matrix 
eigenvalues, eigenvectors = np.linalg.eig(A.todense())
print("Eigenvalues of the adjacency matrix:")
print(eigenvalues)
print("Eigenvectors of the adjacency matrix:")
print(eigenvectors)
# Print A = O * D * O^T
O = np.concat([eigenvectors[:, i].reshape(-1, 1) for i in range(len(eigenvalues))], axis=1)
print("Orthogonal matrix O (eigenvectors):")
print(O)
# Create diagonal matrix D with eigenvalues
D = np.diag(eigenvalues)
A_reconstructed = O @ D @ O.T
print("Reconstructed adjacency matrix:")
print(A_reconstructed)



In [ ]:
# bootstrap analysis

centrality_measures = calculate_centrality_measures(G_clustered)
print("Centrality measures for the clustered graph:")
for measure, values in centrality_measures.items():
    print(f"{measure}: {values}")

# order the centrality measures and print the top 10 nodes for each measure
for measure, values in centrality_measures.items():
    top_10 = sorted(values.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\nTop 10 nodes for {measure}:")
    for node, centrality in top_10:
        print(f"Node {node}: {centrality:.4f}")



# Some considerations

* The pots are all funerary pots? I think so 
* Then we do not have metadata, such as reconstructing time and space. 
* We would like to infer if two datasets overlaps 
* Note that the steps are not commutative, and only the central part is important 


HOW TO PROCEED? I do not know


However, in order to move from the identification of findspots of material (dots on a map) to networks of interaction, connectivity, and trade (lines on a map), some assumptions are needed. First, we must assume that our typology is meaningful. This means that the groupings of material indeed contain elements that are more similar to each other than to the members of other groups. This assumption is necessary to assess whether two findspots share the same types of material in absolute terms or share a certain degree of similarity of material culture in relative terms. Second, we assume that our chronologies are correct, meaning that certain types of materials were produced and used within a certain period of time. This allows us to assess whether the observed similarity is given meaning through contemporaneity. Both assumptions are essential if we want to draw reliable interpretations from our data.

time averaging reduces the accuracy of interpretations by altering network properties such as size, centrality, and path length.

## III.2 Maximum Entropy
* Can we use the maximum entropy formalism? 
* Can we treat the work-chains as "time-series" of different lengths? 
* Can we use another entropy that gives us more information about the influence of a process to another? es Transfer entropy? 

* **From the meeting** Maximum entropy to find how to reconstruct a work chain

* **From the meeting** Temporal network to find if has been a switch between techniques.
 

## III.3 Bayesian attempt

A possible question is: given a new work-chain, can this be explained by the pdf of dataset rather than the other? However we lack of a model and a theoretical pdf, so we can only use the empirical one. Is this meaningful? Likelihood tests.

Information relevance on the remains? 

### Meeting (29/05/2025)
Can we find a graph distance that can explain the time and the space "bias" in the pots

# Random Walk on the graphs of different cultures and look at the distribution statistics